# Evolve Adaptive-MPT prompts on Synthetic-BBQ (Kaggle, one run)

Running all cells does: evolve prompts on the Evolve set -> compare with the originals on the held-out Select set -> overwrite `adaptive_mpt/prompts.py` (backup `prompts.py.bak`) only if the evolved prompts are not worse on the Select set.

Before running: add this repo and `data/synthetic_bbq/` as Kaggle Datasets, turn on a GPU. After a session timeout, re-run with the same `OUT_DIR`; `--resume` continues from `state.json`.

In [ ]:
import os, sys, subprocess
REPO = '/kaggle/input/adaptive-mpt'          # dataset containing this repo (read-only)
WORK = '/kaggle/working/adaptive-mpt'
if not os.path.exists(WORK):
    subprocess.run(['cp', '-r', REPO, WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
!pip -q install -r requirements.txt   # check that vllm installs on this GPU image; see README of the plan for HF fallback

In [ ]:
# HF token for gated models (Kaggle Secrets -> add secret named HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret:', e)

In [ ]:
DATA_DIR = '/kaggle/input/synthetic-bbq'     # dataset holding data/synthetic_bbq/*.jsonl
OUT_DIR  = '/kaggle/working/evolved/run1'
MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
# mpt_evolve.py overwrites adaptive_mpt/prompts.py itself at the end (add --no_apply to disable, --force_apply to override the Select-set guard)
!python mpt_evolve.py --data_dir $DATA_DIR --out_dir $OUT_DIR --models $MODEL --chat_templates \
    --generations 10 --mut_prob 0.5 --num_gpus 2 --max_model_len 8192 --resume

In [ ]:
# Scores per round (also refreshed on disk after every round while the cell above runs)
from IPython.display import Image, display
for name in ['fitness_per_round.png', 'metrics_per_round.png', 'select_set_summary.png']:
    p = f'{OUT_DIR}/plots/{name}'
    if os.path.exists(p):
        display(Image(p))

In [ ]:
import pandas as pd
pd.read_csv(f'{OUT_DIR}/history.csv').tail(20)

In [ ]:
# What changed in prompts.py, and copy the evolved file to the notebook output
import shutil, json
if os.path.exists('adaptive_mpt/prompts.py.bak'):
    !diff -u adaptive_mpt/prompts.py.bak adaptive_mpt/prompts.py
    shutil.copy('adaptive_mpt/prompts.py', '/kaggle/working/prompts_evolved.py')
    print('saved /kaggle/working/prompts_evolved.py  (copy it over adaptive_mpt/prompts.py in your repo)')
else:
    print('prompts.py was not overwritten (see the Select-set message above); best prompts are in', OUT_DIR + '/best_prompts.json')
print(json.load(open(f'{OUT_DIR}/select_metrics.json')))